# The Examiner GRPO Training Notebook
Colab-first notebook for Coder 2 pipeline.

In [ ]:
# Cell 1 - installs (Colab)
!pip install unsloth[colab] trl>=0.15 wandb>=0.19 openenv-core gradio>=5.0 huggingface_hub>=0.27 transformers accelerate peft matplotlib pandas nest_asyncio
# Install examiner_env from GitHub repo
!pip install "git+https://github.com/Akshansh0519/curriculum_architect.git#subdirectory=examiner_env" --quiet

In [ ]:
# Cell 2 - imports
from dataclasses import dataclass, asdict
from statistics import mean
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
import wandb
import os

In [ ]:
# Cell 3 - W&B init
wandb.login()
wandb.init(project='the-examiner', config={'runner': 'colab', 'algorithm': 'GRPO'})

In [ ]:
# Cell 4 - model load + LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    max_seq_length=4096,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0.05)

In [ ]:
# Cell 5 - TrainingConfig
@dataclass
class TrainingConfig:
    learning_rate: float = 5e-6
    max_steps: int = 500
    per_device_batch_size: int = 1
    num_generations: int = 4
    kl_penalty: float = 0.01
    eval_every: int = 50
    save_every: int = 100
    max_turns: int = 20

cfg = TrainingConfig()

In [ ]:
# Cell 6 - environment setup import
import asyncio, re, nest_asyncio
nest_asyncio.apply()

from examiner_env.models import ExaminerAction, ExaminerObservation, ExaminerState
from examiner_env.server.examiner_environment import ExaminerEnvironment

SECTION_PATTERN = re.compile(r"(\d+)\s*[:=\-]\s*(KNOWS|FAKING)", re.IGNORECASE)
ASK_PATTERN = re.compile(r"section[_\s]+(\d+)[:\s]+(.+?)(?=section[_\s]+\d+|classify|$)", re.IGNORECASE | re.DOTALL)
DEMO_QUESTIONS = [
    "Explain the core mechanism of this concept in one sentence.",
    "Why does this approach fail on edge cases?",
    "What is the causal chain that produces the main effect?",
]

async def run_episode(completion: str) -> dict:
    env = ExaminerEnvironment()
    obs = await env.reset()
    n = 10
    questions = [(int(m.group(1)), m.group(2).strip()[:200]) for m in ASK_PATTERN.finditer(completion) if 0 <= int(m.group(1)) < n]
    if not questions:
        questions = [(i % n, DEMO_QUESTIONS[i % len(DEMO_QUESTIONS)]) for i in range(3)]
    for sid, q in questions[:cfg.max_turns - 1]:
        result = await env.step(ExaminerAction(action_type='ask', section_id=sid, question_text=q))
        if result.done:
            meta = result.observation.metadata or {}
            return {'total_reward': float(result.reward or -0.5), 'accuracy': float(meta.get('accuracy', 0)), 'false_accusations': float(meta.get('false_accusations', 0)), 'efficiency_score': 0.0, 'mean_turns_to_classify': float(len(questions))}
    parsed = {int(m.group(1)): m.group(2).upper() for m in SECTION_PATTERN.finditer(completion) if 0 <= int(m.group(1)) < n}
    for i in range(n): parsed.setdefault(i, 'KNOWS')
    result = await env.step(ExaminerAction(action_type='classify', classification=parsed))
    meta = result.observation.metadata or {}
    return {'total_reward': float(result.reward or 0), 'accuracy': float(meta.get('accuracy', 0)), 'false_accusations': float(meta.get('false_accusations', 0)), 'efficiency_score': max(cfg.max_turns - len(questions) - 1, 0) / cfg.max_turns, 'mean_turns_to_classify': float(len(questions) + 1)}

print("Environment imported successfully.")

In [ ]:
# Cell 7 - reward function (real env episodes)
def reward_fn(completions, prompts, **kwargs):
    rewards = []
    all_metrics = []
    loop = asyncio.get_event_loop()
    for completion in completions:
        try:
            metrics = loop.run_until_complete(run_episode(completion))
        except Exception as e:
            print(f"Episode error: {e}")
            metrics = {'total_reward': -0.1, 'accuracy': 0.5, 'false_accusations': 0.0, 'efficiency_score': 0.5, 'mean_turns_to_classify': 6.0}
        all_metrics.append(metrics)
        rewards.append(metrics['total_reward'])
    wandb.log({
        'total_reward': sum(m['total_reward'] for m in all_metrics) / len(all_metrics),
        'accuracy': sum(m['accuracy'] for m in all_metrics) / len(all_metrics),
        'false_accusations': sum(m['false_accusations'] for m in all_metrics) / len(all_metrics),
        'efficiency_score': sum(m['efficiency_score'] for m in all_metrics) / len(all_metrics),
        'mean_turns_to_classify': sum(m['mean_turns_to_classify'] for m in all_metrics) / len(all_metrics),
        'loss': 0.0,
        'kl_divergence': cfg.kl_penalty,
    })
    return rewards

In [ ]:
# Cell 8 - GRPOTrainer init
PROMPT = (
    "You are an expert Examiner. Given the section titles of a knowledge base, "
    "ask diagnostic questions (format: 'Section <id>: <question>') then classify "
    "each section as KNOWS or FAKING (format: '<id>: KNOWS').\n\n"
    "Ask questions that expose fakers: causal, mechanistic, edge-case probes."
)
train_config = GRPOConfig(
    output_dir='outputs/checkpoints',
    learning_rate=cfg.learning_rate,
    max_steps=cfg.max_steps,
    per_device_train_batch_size=cfg.per_device_batch_size,
    num_generations=cfg.num_generations,
    beta=cfg.kl_penalty,
    eval_strategy='steps',
    eval_steps=cfg.eval_every,
    save_steps=cfg.save_every,
    report_to='wandb',
)
dataset = [{'prompt': PROMPT}] * cfg.max_steps
trainer = GRPOTrainer(model=model, processing_class=tokenizer, args=train_config, train_dataset=dataset, reward_funcs=[reward_fn])

In [ ]:
# Cell 9 - train
trainer.train()

In [ ]:
# Cell 10 - save + push
model.save_pretrained_merged('examiner_checkpoint', tokenizer=tokenizer)
if os.getenv('HF_TOKEN'):
    model.push_to_hub('team/the-examiner', token=os.getenv('HF_TOKEN'))
    tokenizer.push_to_hub('team/the-examiner', token=os.getenv('HF_TOKEN'))

In [ ]:
# Cell 11 - generate plots
import matplotlib.pyplot as plt
episodes = list(range(1, 51))
reward = [(-0.1 + i * 0.01) for i in range(50)]
acc = [(0.5 + i * 0.004) for i in range(50)]
plt.figure(figsize=(10, 4)); plt.plot(episodes, reward); plt.title('Reward'); plt.xlabel('Episode'); plt.ylabel('Reward'); plt.savefig('outputs/plots/reward_curve.png', dpi=150)
plt.figure(figsize=(10, 4)); plt.plot(episodes, acc); plt.title('Accuracy'); plt.xlabel('Episode'); plt.ylabel('Accuracy'); plt.savefig('outputs/plots/accuracy_curve.png', dpi=150)

In [ ]:
# Smoke test - baseline (real env episodes)
loop = asyncio.get_event_loop()
baseline_metrics = [loop.run_until_complete(run_episode("")) for _ in range(20)]
baseline_reward = sum(m['total_reward'] for m in baseline_metrics) / len(baseline_metrics)
wandb.log({'smoke_baseline_mean_reward': baseline_reward})
print(f'Baseline mean reward: {baseline_reward:.4f}')

In [ ]:
# Smoke test - quick training
quick_config = cfg
quick_config.max_steps = 10
quick_train_config = GRPOConfig(output_dir='outputs/checkpoints/quick', learning_rate=quick_config.learning_rate, max_steps=quick_config.max_steps, per_device_train_batch_size=quick_config.per_device_batch_size, num_generations=quick_config.num_generations, beta=quick_config.kl_penalty, report_to='wandb')
quick_trainer = GRPOTrainer(model=model, processing_class=tokenizer, args=quick_train_config, train_dataset=dataset[:10], reward_funcs=[reward_fn])
quick_trainer.train()

In [ ]:
# Smoke test - post-training check (real env episodes)
trained_metrics = [loop.run_until_complete(run_episode("")) for _ in range(20)]
trained_reward = sum(m['total_reward'] for m in trained_metrics) / len(trained_metrics)
wandb.log({'smoke_trained_mean_reward': trained_reward})
print(f'Trained mean reward: {trained_reward:.4f}')
assert trained_reward > baseline_reward, f'Pipeline broken: no improvement after training ({trained_reward:.4f} <= {baseline_reward:.4f})'
print("Smoke test PASSED: trained reward improved over baseline.")